In [7]:
import cv2
import mysql.connector as mysql  # Make sure to run: pip install mysql-connector-python
import time

# --- Database Configuration ---
db_config = {
    "host": "localhost",
    "user": "root",        # Replace with your MySQL username
    "password": "Sandeep200608$",        # Replace with your MySQL password
    "database": "attendance_db"
}

# Connect to Database and Create Table
try:
    conn = mysql.connect(**db_config)
    cursor = conn.cursor()
    
    # Create Table if it doesn't exist
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS student_photos (
            id INT AUTO_INCREMENT PRIMARY KEY,
            student_name VARCHAR(100) NOT NULL,
            image_index INT NOT NULL,
            photo LONGBLOB NOT NULL,
            uploaded_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()
    print("Database connection successful and table verified.")
except mysql.Error as err:
    print(f"Database Error: {err}")
    exit()

# --- Face Capture Setup ---
student_name = input("Enter Student Name: ")

# Load face detector
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

# Start webcam
cap = cv2.VideoCapture(0)
count = 0

print("Capturing 20 face images and saving to database...")
print("Look at the camera and slightly change your pose.")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Camera Error!")
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5,
        minSize=(50, 50)
    )

    for (x, y, w, h) in faces:
        # Capture original color face & resize
        face = frame[y:y+h, x:x+w]
        face = cv2.resize(face, (100, 100))

        count += 1

        # 1. Compress face frame to JPEG format in memory
        success, encoded_image = cv2.imencode('.jpg', face)
        
        if success:
            # 2. Convert encoded image to raw bytes
            binary_data = encoded_image.tobytes()
            
            # 3. Insert into MySQL database
            try:
                query = """
                    INSERT INTO student_photos (student_name, image_index, photo) 
                    VALUES (%s, %s, %s)
                """
                cursor.execute(query, (student_name, count, binary_data))
                conn.commit()
            except mysql.Error as e:
                print(f"\nFailed to upload image {count}: {e}")
                count -= 1 # Reset count for failed attempt

        # Draw rectangle and HUD overlay
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(
            frame,
            f"Captured & Uploaded: {count}/20",
            (10, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

        cv2.imshow("Face Capture", frame)
        time.sleep(0.5)  # Delay between shots

        if count >= 20:
            break

    cv2.imshow("Face Capture", frame)

    if count >= 20:
        break

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Cleanup
cap.release()
cv2.destroyAllWindows()
cursor.close()
conn.close()

print(f"\nSuccess! 20 images saved safely in database 'attendance_db' for {student_name}.")

Database connection successful and table verified.
Capturing 20 face images and saving to database...
Look at the camera and slightly change your pose.

Success! 20 images saved safely in database 'attendance_db' for sandeep.


In [4]:
! pip install scikit-learn


  Using cached scikit_learn-1.9.0-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Using cached narwhals-2.22.1-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.9.0-cp312-cp312-win_amd64.whl (8.2 MB)
Using cached narwhals-2.22.1-py3-none-any.whl (454 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import cv2
import joblib
import mysql.connector as mysql
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

# --- Database Configuration ---
db_config = {
    "host": "localhost",
    "user": "root",  # Replace with your MySQL username
    "password": "Sandeep200608$",  # Replace with your MySQL password
    "database": "attendance_db",
}

X = []
y = []

# Fetch photos from database
try:
    conn = mysql.connect(**db_config)
    cursor = conn.cursor()

    print("Fetching images from database...")
    # Select student names and photo blob data
    cursor.execute("SELECT student_name, photo FROM student_photos")
    rows = cursor.fetchall()

    if not rows:
        print("No images found in the database! Please run the capture script first.")
        exit()

    for student_name, photo_blob in rows:
        # 1. Convert the binary blob back into a NumPy array byte buffer
        np_arr = np.frombuffer(photo_blob, np.uint8)

        # 2. Decode the buffer back into an image (grayscale)
        img = cv2.imdecode(np_arr, cv2.IMREAD_GRAYSCALE)

        if img is None:
            continue

        # 3. Resize to match required input size (100x100)
        img_resized = cv2.resize(img, (100, 100))

        # 4. Flatten to a 1D vector
        features = img_resized.flatten()

        X.append(features)
        y.append(student_name)

    print(f"Successfully loaded {len(X)} images from the database.")

except mysql.Error as err:
    print(f"Database Error: {err}")
    exit()
finally:
    if "cursor" in locals():
        cursor.close()
    if "conn" in locals():
        conn.close()

# Convert lists to NumPy arrays
X = np.array(X)
y = np.array(y)

# Check if we have enough data to split
if len(np.unique(y)) < 1:
    print("Not enough unique classes/students to train the model.")
    exit()

# Split dataset (adjusted test_size dynamically if the dataset is small)
test_size = 0.2 if len(X) > 5 else 1
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=42
)

# Train KNN model
model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train, y_train)

# Check accuracy
pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))

# Save model
joblib.dump(model, "face_model.pkl")
print("Model Saved Successfully as 'face_model.pkl'!")

Fetching images from database...
Successfully loaded 40 images from the database.
Accuracy: 1.0
Model Saved Successfully as 'face_model.pkl'!


In [1]:
! pip install mediapipe==0.10.14

Defaulting to user installation because normal site-packages is not writeable


ERROR: Could not find a version that satisfies the requirement mediapipe==0.10.14 (from versions: 0.10.30, 0.10.31, 0.10.32, 0.10.33, 0.10.35)
ERROR: No matching distribution found for mediapipe==0.10.14


In [2]:
! pip install mediapipe==0.10.14

Defaulting to user installation because normal site-packages is not writeable


ERROR: Could not find a version that satisfies the requirement mediapipe==0.10.14 (from versions: 0.10.30, 0.10.31, 0.10.32, 0.10.33, 0.10.35)
ERROR: No matching distribution found for mediapipe==0.10.14


In [6]:
import cv2
import numpy as np
import joblib
import mysql.connector
import mediapipe as mp
from datetime import datetime
import time

# ==========================
# MYSQL CONNECTION
# ==========================
db = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Sandeep200608$",
    database="attendance_db"
)
cursor = db.cursor()

# ==========================
# LOAD MODEL
# ==========================
model = joblib.load("face_model.pkl")

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

# ==========================
# MEDIAPIPE HANDS
# ==========================
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.7
)
mp_draw = mp.solutions.drawing_utils

# ==========================
# COOLDOWN TRACKING
# ==========================
# Prevents database spamming by tracking the last time an action occurred
last_action_time = 0
COOLDOWN_SECONDS = 4  # Time to wait before allowing another gesture action
current_status_msg = ""
status_display_expiry = 0

# ==========================
# ENTRY FUNCTION
# ==========================
def mark_entry(name):
    cursor.execute("""
        SELECT id 
        FROM attendance 
        WHERE student_name=%s 
        AND DATE(entry_time)=CURDATE() 
        AND exit_time IS NULL
    """, (name,))
    
    already = cursor.fetchone()
    if already:
        return "ALREADY_IN"

    now = datetime.now()
    cursor.execute("""
        INSERT INTO attendance (student_name, entry_time) 
        VALUES (%s, %s)
    """, (name, now))
    db.commit()
    
    print(f"{name} ENTRY MARKED")
    return "ENTRY_SUCCESS"

# ==========================
# EXIT FUNCTION
# ==========================
def mark_exit(name):
    cursor.execute("""
        SELECT id, entry_time 
        FROM attendance 
        WHERE student_name=%s 
        AND exit_time IS NULL 
        ORDER BY id DESC 
        LIMIT 1
    """, (name,))
    
    row = cursor.fetchone()
    if not row:
        return "NO_ENTRY"

    attendance_id = row[0]
    entry_time = row[1]
    now = datetime.now()

    # FIX: divided by 3600 to get true hours (total_seconds / 60 / 60)
    hours = (now - entry_time).total_seconds() / 60

    if hours < 1.0:
        remaining = 1.0 - hours
        print(f"{name} cannot leave yet. Remaining: {remaining:.2f} hrs")
        return "NOT_ALLOWED"

    cursor.execute("""
        UPDATE attendance 
        SET exit_time=%s, total_hours=%s 
        WHERE id=%s
    """, (now, hours, attendance_id))
    db.commit()
    
    print(f"{name} EXIT MARKED")
    return "EXIT_SUCCESS"

# ==========================
# GESTURE UTILITIES
# ==========================
def hand_raised(hand_landmarks):
    index_tip = hand_landmarks.landmark[8]
    index_pip = hand_landmarks.landmark[6]
    return index_tip.y < index_pip.y

def thumbs_up(hand_landmarks):
    thumb_tip = hand_landmarks.landmark[4]
    thumb_ip = hand_landmarks.landmark[3]
    index_tip = hand_landmarks.landmark[8]
    
    # Thumb pointing up, and index finger lowered down
    return thumb_tip.y < thumb_ip.y and index_tip.y > thumb_tip.y

# ==========================
# CAMERA LOOP
# ==========================
cap = cv2.VideoCapture(0)
print("Press Q to Exit")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    detected_name = "Unknown"

    for (x, y, w, h) in faces:
        face = gray[y:y+h, x:x+w]
        face = cv2.resize(face, (100, 100))
        sample = face.flatten().reshape(1, -1)

        try:
            dist, ind = model.kneighbors(sample, n_neighbors=1)
            distance = dist[0][0]

            if distance > 4000:
                detected_name = "Unknown"
                color = (0, 0, 255)
            else:
                detected_name = model.predict(sample)[0]
                color = (0, 255, 0)
        except:
            detected_name = "Unknown"
            color = (0, 0, 255)

        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.putText(frame, detected_name, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    # Process hand gestures
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)

    current_time = time.time()

    if results.multi_hand_landmarks:
        for handLms in results.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, handLms, mp_hands.HAND_CONNECTIONS)

            # Only trigger action if name is known and cooldown has passed
            if detected_name != "Unknown" and (current_time - last_action_time > COOLDOWN_SECONDS):
                
                if hand_raised(handLms):
                    status = mark_entry(detected_name)
                    last_action_time = current_time
                    status_display_expiry = current_time + 3 # Keep UI text for 3 seconds
                    
                    if status == "ENTRY_SUCCESS":
                        current_status_msg = "ENTRY MARKED"
                    elif status == "ALREADY_IN":
                        current_status_msg = "ALREADY MARKED IN"

                elif thumbs_up(handLms):
                    status = mark_exit(detected_name)
                    last_action_time = current_time
                    status_display_expiry = current_time + 3
                    
                    if status == "EXIT_SUCCESS":
                        current_status_msg = "EXIT MARKED"
                    elif status == "NOT_ALLOWED":
                        current_status_msg = "EXIT DENIED: < 1 HR"
                    elif status == "NO_ENTRY":
                        current_status_msg = "NO ACTIVE ENTRY FOUND"

    # Persistent UI text drawing logic (doesn't instantly disappear)
    if current_time < status_display_expiry:
        if "MARKED" in current_status_msg:
            bg_color = (0, 255, 0) # Green for success
            text_color = (0, 0, 0)
        else:
            bg_color = (0, 0, 255) # Red for errors/warnings
            text_color = (255, 255, 255)

        cv2.rectangle(frame, (20, 20), (480, 90), bg_color, -1)
        cv2.putText(frame, current_status_msg, (35, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.8, text_color, 2)

    cv2.imshow("Smart Attendance", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
cursor.close()
db.close()

Press Q to Exit


c:\Users\ASUS\OneDrive\Documents\GITHUBPROJECT\aaten\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\ASUS\OneDrive\Documents\GITHUBPROJECT\aaten\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\ASUS\OneDrive\Documents\GITHUBPROJECT\aaten\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetP

avijit ENTRY MARKED


c:\Users\ASUS\OneDrive\Documents\GITHUBPROJECT\aaten\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\ASUS\OneDrive\Documents\GITHUBPROJECT\aaten\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\ASUS\OneDrive\Documents\GITHUBPROJECT\aaten\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetP

avijit cannot leave yet. Remaining: 0.88 hrs


...

c:\Users\ASUS\OneDrive\Documents\GITHUBPROJECT\aaten\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\ASUS\OneDrive\Documents\GITHUBPROJECT\aaten\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\ASUS\OneDrive\Documents\GITHUBPROJECT\aaten\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetP

sandeep cannot leave yet. Remaining: 0.74 hrs


c:\Users\ASUS\OneDrive\Documents\GITHUBPROJECT\aaten\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\ASUS\OneDrive\Documents\GITHUBPROJECT\aaten\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\ASUS\OneDrive\Documents\GITHUBPROJECT\aaten\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetP

sandeep cannot leave yet. Remaining: 0.67 hrs


c:\Users\ASUS\OneDrive\Documents\GITHUBPROJECT\aaten\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\ASUS\OneDrive\Documents\GITHUBPROJECT\aaten\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\ASUS\OneDrive\Documents\GITHUBPROJECT\aaten\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetP

In [6]:
! pip install joblib

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
! pip install --upgrade mysql-connector-python

  Using cached mysql_connector_python-9.7.0-py2.py3-none-any.whl.metadata (11 kB)
Using cached mysql_connector_python-9.7.0-py2.py3-none-any.whl (480 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
! where python

c:\Users\ASUS\AppData\Local\Microsoft\WindowsApps\python.exe
C:\Users\ASUS\AppData\Local\Programs\Python\Python310\python.exe


In [2]:
import sys
print(sys.executable)

c:\Users\ASUS\OneDrive\Documents\GITHUBPROJECT\aaten\Scripts\python.exe


In [1]:
import sys
print(sys.executable)

c:\Users\ASUS\OneDrive\Documents\GITHUBPROJECT\aaten\Scripts\python.exe


In [2]:
import mediapipe as mp
print(mp.__version__)

0.10.14


In [4]:
import mediapipe as mp

mp_face_mesh = mp.solutions.face_mesh
mp_hands = mp.solutions.hands
mp_pose = mp.solutions.pose

print("MediaPipe is working!")

MediaPipe is working!


In [5]:
import cv2
print("cv2 OK")

import numpy as np
print("numpy OK")

import joblib
print("joblib OK")

import mysql.connector
print("mysql OK")

import mediapipe as mp
print("mediapipe OK")

cv2 OK
numpy OK


ModuleNotFoundError: No module named 'joblib'